# 01 – Biztonságkritikus rendszerek, megbízhatóság és kockázat

**1. tanegység · javasolt idő: 2 × 50 perc**

## Tanulási célok

- hazard, risk, fault és failure fogalmak elkülönítése
- megbízhatóság és rendelkezésre állás számítása
- soros és párhuzamos rendszer értelmezése
- a TK-101 veszélylánc első felépítése

**PUNDIT kapcsolat.** Ez a tanegység a PUNDIT projekt (`2020-1.2.3-EUREKA-2022-00021`) eredményeinek **oktatási hasznosítását** támogatja; önmagában nem helyettesíti és nem igazolja a projekt eredeti műszaki vállalásainak teljesítését.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Projekt gyökér:", ROOT)


## 1. Mi a mérnöki kérdés?

Egy safety-critical rendszerben nem elég azt kérdezni, hogy működik-e. Négy kérdésből indulunk: **mi romolhat el, milyen valószínűséggel, mi a következmény, és mivel csökkenthető a kockázat?**

Munkadefiníciók:
- **fault:** hibaok vagy hibaállapot;
- **error:** eltérés a kívánt belső állapottól;
- **failure:** a kívánt funkció elvesztése;
- **hazard:** potenciálisan veszélyes állapot;
- **risk:** a bizonytalan események és következményeik mérnöki jellemzése.

Egyszerű oktatási közelítés: $R=P(H)C$. Ez nem univerzális definíció, hanem belépő modell.


## 2. Megbízhatóság kézzel

Legyen $T$ a meghibásodásig eltelt idő. $R(t)=P(T>t)$ és $F(t)=1-R(t)$. Állandó meghibásodási intenzitás esetén:

$$R(t)=e^{-\lambda t},\qquad F(t)=1-e^{-\lambda t},\qquad MTTF=1/\lambda.$$

Számítsuk ki $\lambda=2\cdot10^{-4}\,h^{-1}$ mellett az 1000 órás megbízhatóságot.


In [ ]:
from math import exp
failure_rate = 2e-4
t = 1000
R_t = exp(-failure_rate*t)
F_t = 1-R_t
print(f"R({t} h)={R_t:.4f}")
print(f"F({t} h)={F_t:.4f}")
print(f"MTTF={1/failure_rate:.0f} h")


### Mit jelent az eredmény?

Az exponenciális modell **memoryless** és állandó $\lambda$-t feltételez. Kopás, öregedés vagy periodikus terhelés esetén ez lehet hibás modellfeltevés. A képlet használata előtt ezért mindig a modellérvényességet kell vizsgálni.


In [ ]:
times=np.linspace(0,5000,200)
for lam in [1e-4,2e-4,5e-4]:
    plt.plot(times,np.exp(-lam*times),label=f"lambda={lam:g} 1/h")
plt.xlabel("Idő [h]"); plt.ylabel("R(t)"); plt.grid(alpha=.3); plt.legend(); plt.show()


## 3. Javítható rendszer és rendelkezésre állás

Kétállapotú exponenciális hiba–javítás modellben $A_\infty=\mu/(\lambda+\mu)$. MTTF/MTTR alakban közelítőleg:

$$A=\frac{MTTF}{MTTF+MTTR}.$$


In [ ]:
MTTF=2000
MTTR=8
A=MTTF/(MTTF+MTTR)
print(f"Availability={A:.6f} = {A*100:.3f}%")
print(f"Várható nem-rendelkezésre állás évente={(1-A)*8760:.1f} h")


## 4. Soros és párhuzamos szerkezet

Független komponenseknél: $R_s=R_1R_2$, míg két párhuzamos, azonos funkciót ellátó elemnél $R_p=1-(1-R_1)(1-R_2)$. A párhuzamos képlet csak akkor írja le jól a redundanciát, ha a hibafüggőségeket nem hagyjuk figyelmen kívül.


In [ ]:
R1,R2=.98,.97
pd.DataFrame({"struktúra":["soros","párhuzamos"],"R":[R1*R2,1-(1-R1)*(1-R2)]})


## 5. TK-101: első veszélylánc

A közös esettanulmány egy éghető folyadék fogadó- és tárolórendszere. Fő elemek: **TK-101 tartály, LT-101 szinttávadó, LSHH-101 védelem, XV-101 elzáró, HMI/operator, CMMS/karbantartás, containment és tűzvédelem**.

Első narratív veszélylánc:

**szenzor degradáció → karbantartás elhalasztása → védelem degradáció → high-level challenge → leállítási/beavatkozási hiba → túlfolyás → gyújtás → tűz.**


In [ ]:
p_challenge=.08
p_auto_fail=.015
p_manual_fail=.0592
p_overflow=p_challenge*p_auto_fail*p_manual_fail
print(f"Durva, függetlenséget feltételező P(overflow)={p_overflow:.8f}")


## Próbáld ki!

1. Növeld a failure rate-et 5e-4-re és értelmezd R(1000)-et.
2. Számíts két R=0.95 elem soros és párhuzamos konfigurációját.
3. Sorolj fel két okot, amiért a TK-101 durva függetlenségi feltételezése hibás lehet.


## Összefoglalás

A megbízhatóság a funkció teljesítését, a rendelkezésre állás a javíthatóságot is figyelembe veszi; a kockázat pedig valószínűséget és következményt kapcsol össze. A következő tanegységben a **mi romolhat el?** kérdést PHA, FMEA és HAZOP segítségével strukturáljuk.
